<a href="https://colab.research.google.com/github/shayanR10/FIV1/blob/FIV1.2/FIV1(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#prerequisites, run this cell first; make sure to also create a Google Kaggle Account along with Colab secrets named "KAGGLE_USERNAME" , "KAGGLE_SLUG" , and "KAGGLE_KEY" for your Google Kaggle
# username, dataset slug, and custom API key, respectively.

!pip install ftfy regex tqdm -q
!pip install git+https://github.com/openai/CLIP.git -q
!pip install kagglehub -q
!pip install numpy -q
!pip install safetensors -q
!pip install cowsay -q
!pip install pillow -q
!pip install open3d -q
!pip install ultralytics -q
!pip install open_clip_torch -q

In [ ]:
#Kaggle connection setup; data pipeline

import os
from google.colab import userdata
import kagglehub

username = userdata.get('KAGGLE_USERNAME')
slug = userdata.get('KAGGLE_SLUG')
key = userdata.get('KAGGLE_KEY')
os.environ["KAGGLE_USERNAME"] = username
os.environ["KAGGLE_KEY"] = key

print("Authenticating")

try:
    path = kagglehub.dataset_download(f"{username}/{slug}")
    print("Success")
    print(path)
except Exception as e:
    print("Failed.")
    kagglehub.login()

In [ ]:
# sam2 (segment anything model)

import os
from PIL import Image, ImageOps
import numpy as np
from ultralytics import SAM
from google.colab import userdata
import kagglehub

print("Waking up SAM 2...")

kUser = userdata.get('KAGGLE_USERNAME')
kSlug = userdata.get('KAGGLE_SLUG')
datapath = kagglehub.dataset_download(f"{kUser}/{kSlug}")

sam = SAM("sam2.1_t.pt")

maskedoutputdir = os.path.join(os.path.dirname(datapath), "fossil_masked_dataset")
os.makedirs(maskedoutputdir, exist_ok=True)

print(f"Processing images from {datapath}...")

processed_count = 0
skipped_count = 0

for root, dirs, files in os.walk(datapath):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_path = os.path.join(root, file)
            print(f"-> Processing: {file}")

            try:
                results = sam(img_path, verbose=False)

                if results and results[0].masks is not None:
                    mask = results[0].masks.data[0].cpu().numpy()

                    img = Image.open(img_path).convert("RGBA")
                    img_np = np.array(img)

                    mask_resized = Image.fromarray((mask * 255).astype(np.uint8)).resize(img.size, Image.Resampling.NEAREST)
                    mask_np = np.array(mask_resized)

                    img_np[:, :, 3] = mask_np
                    masked_img = Image.fromarray(img_np, "RGBA")

                    rel_path = os.path.relpath(img_path, datapath)
                    base_name = os.path.splitext(rel_path)[0]
                    out_path = os.path.join(maskedoutputdir, base_name + ".png")

                    os.makedirs(os.path.dirname(out_path), exist_ok=True)
                    masked_img.save(out_path)
                    processed_count += 1
                else:

                    skipped_count += 1
            except Exception as e:
                print(f"Error processing {file}: {e}. Skipping...")
                skipped_count += 1
                continue

print(f"\nSAM 2 background stripping complete! Successfully processed {processed_count} images (Skipped/No mask: {skipped_count}).")

In [ ]:
# automated multi view rendering for 3d scans

import open3d as o3d
import numpy as np
import math
import os

def classify_fossil_morphology(mesh_path):
    mesh = o3d.io.read_triangle_mesh(mesh_path)
    bbox = mesh.get_bounding_box()
    extents = bbox.get_extents() # [width, height, depth]
    max_extent = np.max(extents)
    min_extent = np.min(extents)
    aspect_ratio = max_extent / (min_extent + 1e-6)
    vertex_count = len(mesh.vertices)

    if vertex_count < 5000 or (aspect_ratio > 3.0 and max_extent < 0.2):
        return "isolated_element"       # Triggers 32 views (2 tiers)
    elif vertex_count > 25000 or max_extent > 1.0:
        return "whole_skeleton"         # Triggers 64 views (4-tier dome)
    else:
        return "skull_or_vertebrae"     # Triggers 48 views (3 tiers for cavities)

def render_fossil_dataset(mesh_path, output_basedir):
    if not os.path.exists(mesh_path):
        print(f"Mesh path {mesh_path} not found.")
        return

    fossil_type = classify_fossil_morphology(mesh_path)
    print(f"\n[3D PIPELINE] Processing asset: {os.path.basename(mesh_path)}")
    print(f"-> Automatically classified morphology as: [{fossil_type.upper()}]")

    if fossil_type == "isolated_element":
        num_views = 32
        elevations = [math.radians(15), math.radians(45)]
        azimuth_steps = 16
    elif fossil_type == "skull_or_vertebrae":
        num_views = 48
        elevations = [math.radians(0), math.radians(30), math.radians(60)]
        azimuth_steps = 16
    elif fossil_type == "whole_skeleton":
        num_views = 64
        elevations = [math.radians(-15), math.radians(15), math.radians(45), math.radians(75)]
        azimuth_steps = 16

    specimen_name = os.path.splitext(os.path.basename(mesh_path))[0]
    output_dir = os.path.join(output_basedir, fossil_type, specimen_name)
    os.makedirs(output_dir, exist_ok=True)

    mesh = o3d.io.read_triangle_mesh(mesh_path)
    mesh.compute_vertex_normals()

    renderer = o3d.visualization.rendering.OffscreenRenderer(224, 224)
    renderer.scene.add_geometry("fossil_mesh", mesh, o3d.visualization.rendering.MaterialRecord())

    bbox = mesh.get_bounding_box()
    center = bbox.get_center()
    radius = np.linalg.norm(bbox.get_max_bound() - bbox.get_min_bound()) / 2.0

    view_idx = 0
    for elevation in elevations:
        for step in range(azimuth_steps):

            azimuth = (step / azimuth_steps) * 2 * math.pi

            cam_x = center[0] + radius * 2.5 * math.cos(elevation) * math.cos(azimuth)
            cam_y = center[1] + radius * 2.5 * math.sin(elevation)
            cam_z = center[2] + radius * 2.5 * math.cos(elevation) * math.sin(azimuth)

            eye = [cam_x, cam_y, cam_z]
            up = [0, 1, 0]

            renderer.setup_camera(60.0, center, eye, up)

            depth_img = renderer.render_to_depth_image(enforce_direct_state=True)
            spatial_img = renderer.render_to_image()

            o3d.io.write_image(f"{output_dir}/view_{view_idx:02d}_depth.png", depth_img)
            o3d.io.write_image(f"{output_dir}/view_{view_idx:02d}_spatial.png", spatial_img)
            view_idx += 1

    print(f"-> Successfully rendered {view_idx} spatial/depth maps to: {output_dir}")

In [ ]:
#image acquisition and upload pipeline

import os
import time
import requests
import torch
import clip
from urllib.parse import urlparse
from google.colab import userdata
from PIL import Image
import kagglehub
from concurrent.futures import ThreadPoolExecutor

# setup
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
kUser = userdata.get('KAGGLE_USERNAME')
kSlug = userdata.get('KAGGLE_SLUG')
dataDir = "./fossil-data"
os.makedirs(dataDir, exist_ok=True)

# Core species list
nonNegotiableDinos = [
    # Stegosauridae
    "Stegosaurus", "Kentrosaurus", "Hesperosaurus", "Tuojiangosaurus",
    "Miragaia", "Dacentrurus",

    # Hadrosauridae & Relatives
    "Edmontosaurus", "Saurolophus", "Maiasaura", "Brachylophosaurus",
    "Gryposaurus", "Prosaurolophus", "Shantungosaurus", "Parasaurolophus",
    "Corythosaurus", "Lambeosaurus", "Hypacrosaurus", "Olorotitan",
    "Tsintaosaurus", "Dryosaurus", "Ouranosaurus", "Iguanodon",
    "Mantellisaurus", "Protohadros", "Probactrosaurus", "Bactrosaurus",
    "Eolambia", "Tethyshadros", "Telmatosaurus",

    # Sauropoda
    "Diplodocus", "Apatosaurus", "Brontosaurus", "Dicraeosaurus",
    "Amargasaurus", "Camarasaurus", "Giraffatitan", "Brachiosaurus",
    "Patagotitan", "Dreadnoughtus", "Rapetosaurus", "Saltasaurus",
    "Futalognkosaurus", "Shunosaurus", "Mamenchisaurus",

    # Ankylosauridae
    "Ankylosaurus", "Zuul", "Euoplocephalus", "Pinacosaurus",
    "Saichania", "Tarchia", "Anodontosaurus", "Scolosaurus",
    "Akainacephalus", "Gobisaurus", "Shamosaurus", "Jinyunpelta",

    # Spinosauridae
    "Baryonyx", "Suchomimus", "Spinosaurus", "Ceratosuchops",
    "Riparovenator", "Irritator", "Spinomimus",

    # Dromaeosauridae & Troodontidae
    "Dromaeosaurus", "Utahraptor", "Deinonychus", "Achillobator",
    "Velociraptor", "Tsaagan", "Linheraptor", "Microraptor",
    "Graciliraptor", "Sinornithosaurus", "Changyuraptor", "Buitreraptor",
    "Halszkaraptor", "Sinornithoides", "Gobivenator", "Mei",

    # Other Major Theropoda
    "Coelophysis", "Procompsognathus", "Dilophosaurus", "Ceratosaurus",
    "Carnotaurus", "Majungasaurus", "Megaraptor", "Allosaurus",
    "Sinraptor", "Monolophosaurus", "Acrocanthosaurus", "Giganotosaurus",
    "Carcharodontosaurus", "Tyrannosaurus", "Tarbosaurus", "Albertosaurus",
    "Gorgosaurus", "Daspletosaurus", "Yutyrannus", "Guanlong",
    "Compsognathus", "Sinosauropteryx", "Sinocalliopteryx", "Huaxiagnathus",
    "Gallimimus", "Struthiomimus", "Ornithomimus", "Therizinosaurus",
    "Alxasaurus", "Erlikosaurus", "Anzu", "Citipati", "Khaan",
    "Conchoraptor", "Caudipteryx",

    # Ceratopsia
    "Triceratops", "Torosaurus", "Chasmosaurus", "Anchiceratops",
    "Arrhinoceratops", "Kosmoceratops", "Utahceratops", "Pentaceratops",
    "Centrosaurus", "Styracosaurus", "Pachyrhinosaurus", "Nasutoceratops",
    "Einiosaurus", "Diabloceratops", "Wendiceratops", "Medusaceratops",
    "Sinoceratops", "Protoceratops", "Psittacosaurus"
]

mIMperSP = 200
maxTotalSpecies = len(nonNegotiableDinos) + 100

def getDINOS(limit):
    print("Building hybrid species list...")
    dinoList = list(nonNegotiableDinos)

    if len(dinoList) >= limit:
        return dinoList[:limit]

    wikiURL = "https://en.wikipedia.org/w/api.php"
    headers = {
        "User-Agent": f"dinodatafinder/1.0 (https://kaggle.com/{kUser}) Python-requests"
    }
    params = {
        "action": "query",
        "list": "categorymembers",
        "cmtitle": "Category:Dinosaur_genera",
        "cmlimit": 500,
        "cmtype": "page",
        "format": "json"
    }

    resp = requests.get(wikiURL, params=params, headers=headers)
    resp.raise_for_status()
    rawList = [page['title'] for page in resp.json()['query']['categorymembers']]

    excludeWords = [
        "acta", "journal", "annals", "bulletin", "list", "formation",
        "museum", "paleont", "suidae", "family", "cf.", "indet",
        "<span>", "http", "doi"
    ]

    for name in rawList:
        nameClean = name.strip()
        if not nameClean:
            continue

        parts = nameClean.split()

        # Grab only the first word and capitalize it
        genusName = parts[0].capitalize()

        # Skip if the first character isn't a letter
        if not genusName[0].isalpha():
            continue

        # Check the original full string for exclude words and taxonomy endings
        lowerName = nameClean.lower()
        if any(word in lowerName for word in excludeWords):
            continue
        if lowerName.endswith("idae") or lowerName.endswith("inae") or lowerName.endswith("osauria"):
            continue

        # add to list if it's not already there
        if genusName not in dinoList:
            dinoList.append(genusName)
            if len(dinoList) >= limit:
                break

    print(f"Total target species: {len(dinoList)}")
    return dinoList

speciesList = getDINOS(maxTotalSpecies)

print("loading CLIP")
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)
print("CLIP loaded successfully.")

junkTags = [
    'toy', 'plastic', 'figure', 'game', 'render', '3d', 'plush',
    'sculpture', 'lego', 'meme', 'origami', 'suit', 'costume',
    'animatronic', 'illustration', 'drawing', 'painting', 'cartoon',
    'diorama', 'taxidermy', 'human', 'tourist', 'graffiti', 'logo'
]

import os
import time

def downloadSingleImage(imgUrl, imgPath, session, maxsizemb=15):
    try:
        # skip downloading if already cached
        if os.path.exists(imgPath):
            return True

        time.sleep(0.1)
        # use stream=True to inspect headers
        with session.get(imgUrl, timeout=(5, 10), stream=True) as imgResp:
            if imgResp.status_code == 200:
                max_bytes = maxsizemb * 1024 * 1024

                # Check header if present
                file_size = int(imgResp.headers.get('Content-Length', 0))
                if file_size > max_bytes:
                    print(f"warning: skipping oversized file header ({file_size / 1024 / 1024:.2f} MB): {imgUrl}")
                    return False

                # guards against missing headers
                downloaded_bytes = 0
                with open(imgPath, "wb") as f:
                    for chunk in imgResp.iter_content(chunk_size=8192):
                        if chunk:
                            downloaded_bytes += len(chunk)
                            if downloaded_bytes > max_bytes:
                                print(f"warning: aborted image exceeding {maxsizemb}MB limit: {imgUrl}")
                                f.close()
                                if os.path.exists(imgPath):
                                    os.remove(imgPath) # clean up partial files
                                return False
                            f.write(chunk)
                return True
    except Exception as e:
        pass
    return False

def scrapeDino(species, maxRes):
    folderName = species.replace(" ", "_")
    fPath = os.path.join(dataDir, folderName)
    os.makedirs(fPath, exist_ok=True)

    existFiles = [f for f in os.listdir(fPath) if f.endswith('.jpg')]
    if len(existFiles) >= maxRes:
        print(f"\n[{species}] Checkpoint active: {len(existFiles)} images found. Skipping download.")
        return

    print(f"\n[{species}] scraping in progress...")

    url = "https://commons.wikimedia.org/w/api.php"
    headers = {
        "User-Agent": f"dinodatafinder/1.0 (https://kaggle.com/{kUser}) Python-requests",
        "Referer": "https://commons.wikimedia.org/"
    }

    session = requests.Session()
    session.headers.update(headers)

    saved = len(existFiles)
    skipped = 0
    searchQueries = [
      f"{species} skeleton",
      f"{species} fossil",
      f"{species} specimen",
      f"{species} skull",
      f"{species} holotype",
      f"{species} bones",
      f"{species} mount",
      f"{species} museum",
      f"{species}",
  ]
    downloadTasks = []

    try:
        for query in searchQueries:
            if saved + len(downloadTasks) >= maxRes:
                break

            srOffset = 0
            while (saved + len(downloadTasks)) < maxRes and srOffset < 300:
                params = {
                    "action": "query",
                    "generator": "search",
                    "gsrsearch": query,
                    "gsrnamespace": "6",
                    "gsrlimit": 50,
                    "gsroffset": srOffset,
                    "prop": "imageinfo",
                    "iiprop": "url|extmetadata",
                    "format": "json"
                }

                resp = session.get(url, params=params, timeout=10).json()

                if "query" in resp and "pages" in resp["query"]:
                    pages = resp["query"]["pages"]
                    for pId, pInfo in pages.items():
                        if (saved + len(downloadTasks)) >= maxRes:
                            break
                        if "imageinfo" in pInfo:
                            info = pInfo['imageinfo'][0]
                            imgURL = info['url']
                            meta = info.get('extmetadata', {})

                            pathParse = urlparse(imgURL).path.lower()
                            if not any(pathParse.endswith(ext) for ext in ['.jpg', '.jpeg', '.png', '.webp']):
                                continue

                            metaText = ""
                            if 'ImageDescription' in meta:
                                metaText += str(meta['ImageDescription'].get('value', '')).lower()
                            if 'Categories' in meta:
                                metaText += str(meta['Categories'].get('value', '')).lower()

                            if any(j in metaText for j in junkTags):
                                skipped += 1
                                continue

                            targetPath = os.path.join(fPath, f"{saved + len(downloadTasks):03d}.jpg")
                            downloadTasks.append((imgURL, targetPath))
                    srOffset += 50
                else:
                    break

        if downloadTasks:
            with ThreadPoolExecutor(max_workers=10) as executor:
                futures = [executor.submit(downloadSingleImage, imgUrl, targetPath, session) for imgUrl, targetPath in downloadTasks]
                for future in futures:
                    if future.result():
                        saved += 1

        print(f"[{species}] Total valid images: {saved}. Skipped {skipped} via metadata.")
    except Exception as e:
        print(f"Error scraping {species}: {e}")

def filterIM(species):
    folderName = species.replace(" ", "_")
    fPath = os.path.join(dataDir, folderName)
    if not os.path.exists(fPath):
        return

    print(f"[{species}] Filtering images...")
    broken = 0
    badVis = 0

    prompts = [
        f"A clear, isolated photograph of a single complete museum dinosaur skeleton of a {species}",
        "A scientific diagram, taxonomic tree, family tree chart, or text-heavy graphic",
        "An illustration, painting, digital render, or children's book page of a dinosaur",
        "A collage, museum wall layout, or photograph containing multiple different dinosaur skeletons",
        "A close-up macro shot of a single isolated bone, individual tooth, or detached claw"
    ]

    tokens = clip.tokenize(prompts).to(device)
    allFiles = [f for f in os.listdir(fPath) if f.endswith('.jpg')]

    batchSize = 32
    for i in range(0, len(allFiles), batchSize):
        batchFiles = allFiles[i:i + batchSize]
        batchImgs = []
        batchPaths = []

        for fName in batchFiles:
            imgPath = os.path.join(fPath, fName)
            try:
                # file integrity
                with Image.open(imgPath) as test_img:
                    test_img.verify()

                # 2. safely open file
                with Image.open(imgPath) as raw_img:
                    # clear metadata
                    raw_img.info.clear()  # wipe metadata dictionary

                    # convert to RGB, preprocess
                    clean_img = raw_img.convert("RGB")
                    batchImgs.append(preprocess(clean_img))

                batchPaths.append(imgPath)

            except (IOError, SyntaxError, ValueError, RuntimeError) as e:
                # possible malware filter
                print(f"warning: deleting possible malware ({fName}): {e}")
                try:
                    if os.path.exists(imgPath):
                        os.remove(imgPath)
                except Exception:
                    pass
                broken += 1

        if not batchImgs:
            continue

        imageTensor = torch.stack(batchImgs).to(device)

        with torch.no_grad():
            logits, _ = model(imageTensor, tokens)
            probs = logits.softmax(dim=-1).cpu().numpy()

        for idx, p in enumerate(probs):
            if p.argmax() != 0:
                os.remove(batchPaths[idx])
                badVis += 1

    print(f"[{species}] done. Removed {broken} broken, {badVis} visually irrelevant items.")

def kagglePush():
    print("\nprepping upload (kagglehub)")
    try:
        handle = f"{kUser}/{kSlug}"
        print(f"Pushing to Kaggle dataset handle: {handle}")

        kagglehub.dataset_upload(
            handle=handle,
            local_dataset_dir=dataDir,
              version_notes="new set"
        )
        print("Upload complete! Check your Kaggle profile.")
    except Exception as e:
        print(f"Kagglehub upload failed: {e}")

if __name__ == "__main__":
    for dino in speciesList:
        scrapeDino(dino, mIMperSP)
        filterIM(dino)
    kagglePush()

In [ ]:
# CONFIGURATIONS
imagesizing = (224, 224, 3)    # Image sizing requirement
confidencelevel = (0.85, 0.50) # Confidence thresholds: if confidencelevel, the level of confidence the model has -
                               # in its prediction is 0.85/85% or more, it confirms that exact species. -
                               # Otherwise, graceful fallback is triggered (e.g. Ouranosaurus would simply be -
                               # classified as "unidentified basal hadrosauriform" [or simpler if needbe].)

# safeguard-1: confirms tuples containing image sizing and confidence intervals.
def safeguard1():
  print(imagesizing, confidencelevel)

safeguard1()

# requesting data from the Paleobiology Database's (PBDB) live API
import requests
DINO_INFO_url = "https://paleobiodb.org/data1.2/taxa/list.json?base_name=Dinosauria&status=accepted"
def getdinosauria(DINO_INFO_url):
  response = requests.get(DINO_INFO_url, timeout=30)
  response.raise_for_status()
  dinodata = response.json()
  return dinodata

dinodata = getdinosauria(DINO_INFO_url)

# parses through data
dinotax2 = {}
for records in dinodata["records"]:
  txnID = records["oid"]
  dinotax2[txnID] = {
      "name": records.get("nam"),
      "rank": records.get("rnk"),
      "parent": records.get("par")
  }

def fallingback(txnID, allowed=["genus", "subfamily", "family", "superfamily", "infraorder"]):
  records2 = dinotax2.get(txnID)
  if not records2:
    return "Unidentified Dinosauria", "clade"

  parentID = records2.get("parent")

  bestmatchN = "Unidentified Dinosauria"
  bestmatchR = "clade"

  while parentID:
    parentrnk = dinotax2.get(parentID)
    if not parentrnk:
      break

    currentrank = parentrnk.get("rank")
    currentname = parentrnk.get("name")

    if currentrank in allowed:
      bestmatchN = currentname
      bestmatchR = currentrank
      if currentrank == allowed[-1]:
        break

    parentID = parentrnk.get("parent")

  return bestmatchN, bestmatchR


# safeguard
print(f"{len(dinotax2)} taxons identified")

# conditionals concerning previously defined confidence thresholds
highconf = confidencelevel[0]
lowconf = confidencelevel[1]

def guessing(species, confscore, txnID):
  if confscore >= highconf:
    print(f"{confscore * 100:.1f}% confident of {species}")
    return species, confscore, None
  elif confscore >= lowconf:
    fallbackname, fallbackrank = fallingback(txnID)
    print(f"{confscore * 100:.1f}% confident of {fallbackname} in {fallbackrank}")
    return fallbackname, confscore, fallbackrank
  else:
    fallbackname, fallbackrank = fallingback(txnID)
    print(f"{confscore * 100:.1f}% confidence, unable to identify. Defaulting to broad clade: {fallbackname} ({fallbackrank})")
    return None, confscore, fallbackrank


# neural network, image processing
import torch
import torch.nn as neuralnetwork
import torchvision.models as TVmodels
import torchvision.datasets as dataset
from torchvision import transforms
from torch.utils.data import DataLoader, Subset
import kagglehub
import os
import numpy as nump
import random
from collections import defaultdict
import cowsay
from safetensors.torch import save_file
import json
import open_clip

try:
  import cv2
  from PIL import Image, ImageFilter
except ImportError:
  cv2 = None
  from PIL import Image, ImageFilter

try:
  from google.colab import userdata
except ImportError:
  userdata = None


# pulling data from kaggle dataset for the model to process
kUser = None
kSlug = None
if userdata is not None:
  kUser = userdata.get("KAGGLE_USERNAME")
  kSlug = userdata.get("KAGGLE_SLUG")

if kUser and kSlug:
  print("Fetching dataset from Kaggle")
  datapath = kagglehub.dataset_download(f"{kUser}/{kSlug}")
else:
  datapath = os.environ.get("DINO_DATASET_PATH")

if not datapath:
  raise RuntimeError(
      "Set DINO_DATASET_PATH or Kaggle Colab secrets "
      "KAGGLE_USERNAME/KAGGLE_SLUG."
  )

print(f"Data ready at: {datapath}")


# training optimizations to avoid model from memorizing training images
class Stream2NoiseGuard:
  def __init__(self, kernel_size=3, sigma=0.8, enhance=False):
    self.kernel_size = kernel_size
    self.sigma = sigma
    self.enhance = enhance

  def __call__(self, pil_image):
    image = pil_image.convert("RGB")
    image = image.filter(
        ImageFilter.GaussianBlur(
            radius=(self.sigma * (self.kernel_size - 1) / 2)
        )
    )
    if not self.enhance:
      return image
    if cv2 is None:
      print("[WARNING] cv2 unavailable; Stream 2 enhancement is skipped.")
      return image

    array = nump.asarray(image)
    lab = cv2.cvtColor(array, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(8, 8))
    lab = cv2.merge((clahe.apply(l), a, b))
    clahe_rgb = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

    laplacian = cv2.Laplacian(
        cv2.cvtColor(clahe_rgb, cv2.COLOR_RGB2GRAY),
        cv2.CV_32F,
        ksize=3
    )
    laplacian = cv2.normalize(
        laplacian,
        None,
        alpha=0.0,
        beta=1.0,
        norm_type=cv2.NORM_MINMAX
    )
    laplacian_rgb = cv2.cvtColor(
        (laplacian * 255).astype(nump.uint8),
        cv2.COLOR_GRAY2RGB
    )
    blended = nump.clip(0.90 * clahe_rgb + 0.10 * laplacian_rgb, 0, 255)
    return Image.fromarray(blended.astype(nump.uint8))


# ViT stream 1 transforms
macrotransforms = transforms.Compose([
  transforms.Resize((224, 224)),
  transforms.RandomHorizontalFlip(p=0.5),
  transforms.ColorJitter(brightness=0.1, contrast=0.1),
  transforms.ToTensor(),
  transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# micro-texture stream 2
microtransforms = transforms.Compose([
  Stream2NoiseGuard(kernel_size=3, sigma=0.8, enhance=True),
  transforms.Resize((256, 256)),
  transforms.RandomResizedCrop(224, scale=(0.3, 0.6)),
  transforms.RandomHorizontalFlip(p=0.5),
  transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
  transforms.ToTensor(),
  transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
  transforms.RandomErasing(p=0.1, scale=(0.02, 0.08), value="random")
])

# 3d scan transforms
spatialtransforms = transforms.Compose([
  transforms.Resize((224, 224)),
  transforms.ToTensor(),
  transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# val transforms
valmacro = transforms.Compose([
  transforms.Resize((224, 224)),
  transforms.ToTensor(),
  transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
valmicro = transforms.Compose([
  Stream2NoiseGuard(kernel_size=3, sigma=0.8, enhance=False),
  transforms.Resize((256, 256)),
  transforms.CenterCrop(224),
  transforms.ToTensor(),
  transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
valspatial = transforms.Compose([
  transforms.Resize((224, 224)),
  transforms.ToTensor(),
  transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class tripledata(torch.utils.data.Dataset):
  def __init__(self, root_dir, macroTf, microTf, spatialTf):
    self.base_dataset = dataset.ImageFolder(root=root_dir)
    self.classes = self.base_dataset.classes
    self.targets = self.base_dataset.targets
    self.macroTf = macroTf
    self.microTf = microTf
    self.spatialTf = spatialTf

  def __len__(self):
    return len(self.base_dataset)

  def __getitem__(self, idx):
    img, label = self.base_dataset[idx]
    return {
        "macro": self.macroTf(img),
        "micro": self.microTf(img),
        "spatial": self.spatialTf(img),
        "label": label
    }

fulltrainset = tripledata(
    root_dir=datapath,
    macroTf=macrotransforms,
    microTf=microtransforms,
    spatialTf=spatialtransforms
)
valset = tripledata(
    root_dir=datapath,
    macroTf=valmacro,
    microTf=valmicro,
    spatialTf=valspatial
)
speciesamt = len(fulltrainset.classes)

with open("class_names.json", "w") as file1:
  json.dump(fulltrainset.classes, file1)

targetranks = ["family", "genus", "superfamily", "subfamily", "infraorder"]

RECORDname = {}
for taxonID2, record in dinotax2.items():
  if record["name"]:
    RECORDname[record["name"].strip().lower()] = record

HIERARCHY = {}
rankvocabularies = {r: {} for r in targetranks}

for cINDx, cNAME in enumerate(fulltrainset.classes):
  cleaned = cNAME.replace("_", " ").strip().lower()
  record = RECORDname.get(cleaned)

  mapped = {}
  currentR = record
  while currentR:
    rank = currentR.get("rank")
    name = currentR.get("name")
    if rank in targetranks and name:
      if name not in rankvocabularies[rank]:
        rankvocabularies[rank][name] = len(rankvocabularies[rank])
      mapped[rank] = rankvocabularies[rank][name]

    parentID2 = currentR.get("parent")
    currentR = dinotax2.get(parentID2) if parentID2 else None

  HIERARCHY[cINDx] = mapped

# fallback sizes if vocabularies are empty
rank_classes = {r: max(1, len(rankvocabularies[r])) for r in targetranks}
print("Hierarchical Head Classes ->", rank_classes)

# splitting: 80% of images go to training while the rest go to validation testing
datasetSIZE = len(fulltrainset)
targets = fulltrainset.targets

classindices = defaultdict(list)
for index, target in enumerate(targets):
  classindices[target].append(index)

trainindices = []
valindices = []

random.seed(42)
for target, indices in classindices.items():
  random.shuffle(indices)

  if len(indices) == 1:
    trainindices.extend(indices)
  else:
    split = max(1, int(len(indices) * 0.8))
    trainindices.extend(indices[:split])
    valindices.extend(indices[split:])

random.shuffle(trainindices)
random.shuffle(valindices)

training = Subset(fulltrainset, trainindices)
validation = Subset(valset, valindices)

# separate validation model selection from post-hoc temperature calibration
generator = torch.Generator().manual_seed(42)
validation_selection, validation_calibration = torch.utils.data.random_split(
    validation,
    [len(validation) // 2, len(validation) - (len(validation) // 2)],
    generator=generator
)

# another safeguard
print(f"{speciesamt} species found for training")
print(f"Training on {len(training)} images, Validating on {len(validation)} images.")

# data loaders
loader = DataLoader(training, batch_size=16, shuffle=True)
validationL = DataLoader(validation_selection, batch_size=16, shuffle=False)
calibrationL = DataLoader(validation_calibration, batch_size=16, shuffle=False)


# triple stream data loaders
class CrossAttentiveFusion(neuralnetwork.Module):
  def __init__(self, embed_dim=256, num_heads=4):
    super().__init__()
    self.MULTI = neuralnetwork.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)
    self.lNORM = neuralnetwork.LayerNorm(embed_dim)
    self.gating = neuralnetwork.Sequential(
        neuralnetwork.Linear(embed_dim * 3, 3),
        neuralnetwork.Sigmoid()
    )

  def forward(self, z1, z2, z3):
    sequenceTENSOR = torch.stack([z1, z2, z3], dim=1)
    concatflat = torch.cat([z1, z2, z3], dim=1)
    gate_vals = self.gating(concatflat).unsqueeze(-1)
    attenout, _ = self.MULTI(sequenceTENSOR, sequenceTENSOR, sequenceTENSOR)
    attenout = self.lNORM(attenout + sequenceTENSOR)
    gated = attenout * gate_vals
    fused = gated.reshape(gated.shape[0], -1)
    return fused

class MSDhead(neuralnetwork.Module):
  def __init__(self, in_features, numclasses1, numsamples1=5, p=0.2):
    super().__init__()
    self.numsamples1 = numsamples1
    self.layernorm = neuralnetwork.LayerNorm(in_features)
    self.act = neuralnetwork.GELU()
    self.dropouts = neuralnetwork.ModuleList(
        [neuralnetwork.Dropout(p) for _ in range(numsamples1)]
    )
    self.linear = neuralnetwork.Linear(in_features, numclasses1)

  def forward(self, AT):
    AT = self.layernorm(AT)
    AT = self.act(AT)
    if not self.training:
      return self.linear(AT)
    logits = sum(self.linear(drop(AT)) for drop in self.dropouts) / self.numsamples1
    return logits

class OrthogonalStreamContrastiveLoss(neuralnetwork.Module):
  def __init__(self, alpha=0.1):
    super().__init__()
    self.alpha = alpha

  def forward(self, z1, z2, z3):
    eps = 1e-8

    z1norm = z1 / (torch.norm(z1, p=2, dim=1, keepdim=True) + eps)
    z2norm = z2 / (torch.norm(z2, p=2, dim=1, keepdim=True) + eps)
    z3norm = z3 / (torch.norm(z3, p=2, dim=1, keepdim=True) + eps)

    sim12 = torch.sum(z1norm * z2norm, dim=1).abs().mean()
    sim13 = torch.sum(z1norm * z3norm, dim=1).abs().mean()
    sim23 = torch.sum(z2norm * z3norm, dim=1).abs().mean()

    ortholoss = (sim12 + sim13 + sim23) / 3.0
    return self.alpha * ortholoss


class SAM(torch.optim.Optimizer):
  def __init__(self, params, base_optimizer, rho=0.05, **kwargs):
    assert rho >= 0.0, f"Invalid rho, should be non-negative: {rho}"
    param_groups = params if params and isinstance(params[0], dict) else list(params)
    defaults = dict(rho=rho, **kwargs)
    super(SAM, self).__init__(param_groups, defaults)

    self.base_optimizer = base_optimizer(self.param_groups, **kwargs)
    self.param_groups = self.base_optimizer.param_groups

  @torch.no_grad()
  def first_step(self, zero_grad=False):
    grad_norm = self._grad_norm()

    if not torch.isfinite(grad_norm) or grad_norm <= 1e-12:
      if zero_grad:
        self.zero_grad(set_to_none=True)
      return

    grad_norm = torch.clamp(grad_norm, min=1e-12)
    for group in self.param_groups:
      scale = group["rho"] / grad_norm
      for p in group["params"]:
        if p.grad is None:
          continue
        if not torch.isfinite(p.grad).all():
          continue
        self.state[p]["old_p"] = p.detach().clone()
        epsil = p.grad * scale
        p.add_(epsil)

    if zero_grad:
      self.zero_grad(set_to_none=True)

  @torch.no_grad()
  def second_step(self, zero_grad=False):
    for group in self.param_groups:
      for p in group["params"]:
        old_p = self.state[p].pop("old_p", None)
        if old_p is not None:
          p.copy_(old_p)

    self.base_optimizer.step()
    if zero_grad:
      self.zero_grad(set_to_none=True)

  @torch.no_grad()
  def _grad_norm(self):
    shared_device = self.param_groups[0]["params"][0].device
    finite_grads = []

    for group in self.param_groups:
      for p in group["params"]:
        if p.grad is not None and torch.isfinite(p.grad).all():
          finite_grads.append(p.grad.detach().norm(p=2).to(shared_device))

    if not finite_grads:
      return torch.tensor(0.0, device=shared_device)

    grad_norm = torch.norm(torch.stack(finite_grads), p=2)
    return grad_norm if torch.isfinite(grad_norm) else torch.tensor(
        0.0, device=shared_device
    )

  def zero_grad(self, set_to_none=False):
    self.base_optimizer.zero_grad(set_to_none=set_to_none)


def find_bioclip_blocks(visual):
  if hasattr(visual, "transformer") and hasattr(visual.transformer, "resblocks"):
    return list(visual.transformer.resblocks)
  if hasattr(visual, "blocks"):
    return list(visual.blocks)
  if hasattr(visual, "trunk") and hasattr(visual.trunk, "blocks"):
    return list(visual.trunk.blocks)
  return []

def freeze_bioclip_visual_blocks(visual, freeze_fraction=0.75, top_trainable=3):
  blocks = find_bioclip_blocks(visual)
  if not blocks:
    print("[WARNING] BioCLIP transformer blocks not found; no selective freeze applied.")
    for p in visual.parameters():
      p.requires_grad = False
    return 0

  frozen_count = int(len(blocks) * freeze_fraction)
  frozen_count = max(0, min(frozen_count, len(blocks) - top_trainable))
  for p in visual.parameters():
    p.requires_grad = True

  for block in blocks[:frozen_count]:
    for p in block.parameters():
      p.requires_grad = False
  for block in blocks[frozen_count:]:
    for p in block.parameters():
      p.requires_grad = True

  for name in ("ln_post", "proj", "head"):
    module = getattr(visual, name, None)
    if module is not None:
      for p in module.parameters():
        p.requires_grad = True

  frozen = sum(not p.requires_grad for p in visual.parameters())
  trainable = sum(p.requires_grad for p in visual.parameters())
  print(f"BioCLIP visual: froze {frozen_count}/{len(blocks)} bottom blocks "
        f"({trainable} tensors trainable, {frozen} frozen)")
  return len(blocks) - frozen_count

def build_optimizer_param_groups(model, visual, base_lr=1e-5, decay=0.75, weight_decay=0.01):
  blocks = find_bioclip_blocks(visual)
  block_ids = {id(block): index for index, block in enumerate(blocks)}
  decay_groups = defaultdict(list)

  for name, p in model.named_parameters():
    if not p.requires_grad:
      continue

    lr = base_lr
    parent_block = None
    if name.startswith("path1ViT."):
      for block in blocks:
        if any(p is block_p for block_p in block.parameters()):
          parent_block = block
          break

    if parent_block is not None:
      block_index = block_ids[id(parent_block)]
      depth_from_top = len(blocks) - 1 - block_index
      lr = base_lr * (decay ** depth_from_top)

    is_normalized = name.endswith("bias") or "norm" in name.lower() or "ln" in name.lower()
    decay_groups[(lr, is_normalized)].append(p)

  groups = []
  seen_parameters = set()
  for (lr, is_normalized), parameters in sorted(
      decay_groups.items(), key=lambda item: item[0][0], reverse=True
  ):
    unique = []
    for p in parameters:
      if id(p) not in seen_parameters:
        unique.append(p)
        seen_parameters.add(id(p))
    if unique:
      groups.append({
          "params": unique,
          "lr": lr,
          "weight_decay": 0.0 if is_normalized else weight_decay,
          "rho": 0.05
      })

  return groups


class setupForData(neuralnetwork.Module):
  def __init__(self, numclasses, rank_classes_dict, freeze_backbones=False):
    super().__init__()
    print("waking up BioCLIP 2 (imageomics/bioclip-2) macro model")
    bioclip_model, _, _ = open_clip.create_model_and_transforms("hf-hub:imageomics/bioclip-2")
    self.path1ViT = bioclip_model.visual
    self.projection1 = neuralnetwork.Sequential(
        neuralnetwork.Linear(768, 256),
        neuralnetwork.LayerNorm(256),
        neuralnetwork.GELU()
    )

    print("waking up textures model")
    ConvMicro = TVmodels.convnext_tiny(weights=TVmodels.ConvNeXt_Tiny_Weights.DEFAULT)
    self.path2CVN = neuralnetwork.Sequential(
        *list(ConvMicro.children())[:-1],
        neuralnetwork.AdaptiveAvgPool2d(1),
        neuralnetwork.Flatten()
    )
    self.projection2 = neuralnetwork.Sequential(
        neuralnetwork.Linear(768, 256),
        neuralnetwork.LayerNorm(256),
        neuralnetwork.GELU()
    )

    print("waking up spatial model")
    ConvSpace = TVmodels.convnext_tiny(weights=TVmodels.ConvNeXt_Tiny_Weights.DEFAULT)
    self.path3CVN = neuralnetwork.Sequential(
        *list(ConvSpace.children())[:-1],
        neuralnetwork.AdaptiveAvgPool2d(1),
        neuralnetwork.Flatten()
    )
    self.projection3 = neuralnetwork.Sequential(
        neuralnetwork.Linear(768, 256),
        neuralnetwork.LayerNorm(256),
        neuralnetwork.GELU()
    )

    if freeze_backbones:
      print("[INFO] Freezing DINOv2 and ConvNeXt backbones.")
      for p in self.path1ViT.parameters():
        p.requires_grad = False
      for p in self.path2CVN.parameters():
        p.requires_grad = False
      for p in self.path3CVN.parameters():
        p.requires_grad = False
    else:
      freeze_bioclip_visual_blocks(
          self.path1ViT,
          freeze_fraction=0.75,
          top_trainable=3
      )

    self.fusion_neck = CrossAttentiveFusion(embed_dim=256, num_heads=4)

    self.species_head = MSDhead(in_features=768, numclasses1=numclasses + 1, numsamples1=5, p=0.2)
    self.family_head = MSDhead(in_features=768, numclasses1=rank_classes_dict["family"], numsamples1=5, p=0.2)
    self.genus_head = MSDhead(in_features=768, numclasses1=rank_classes_dict["genus"], numsamples1=5, p=0.2)
    self.superfamily_head = MSDhead(in_features=768, numclasses1=rank_classes_dict["superfamily"], numsamples1=5, p=0.2)
    self.subfamily_head = MSDhead(in_features=768, numclasses1=rank_classes_dict["subfamily"], numsamples1=5, p=0.2)
    self.infraorder_head = MSDhead(in_features=768, numclasses1=rank_classes_dict["infraorder"], numsamples1=5, p=0.2)

  def forward(self, macroIN, microIN, spatialIN):
    x1 = self.path1ViT(macroIN)
    z1 = self.projection1(x1)

    x2 = self.path2CVN(microIN)
    z2 = self.projection2(x2)

    x3 = self.path3CVN(spatialIN)
    z3 = self.projection3(x3)

    # Zero-guard to prevent blank spatial inputs from polluting attention
    if spatialIN.abs().sum() == 0:
      z3 = torch.zeros_like(z2)

    fused = self.fusion_neck(z1, z2, z3)

    def safe_logits(logits):
      logits = torch.nan_to_num(
          logits,
          nan=0.0,
          posinf=30.0,
          neginf=-30.0
      )
      return logits.clamp(min=-30.0, max=30.0)

    outputs = {
        "species": safe_logits(self.species_head(fused)),
        "family": safe_logits(self.family_head(fused)),
        "genus": safe_logits(self.genus_head(fused)),
        "superfamily": safe_logits(self.superfamily_head(fused)),
        "subfamily": safe_logits(self.subfamily_head(fused)),
        "infraorder": safe_logits(self.infraorder_head(fused)),
        "z1": z1,
        "z2": z2,
        "z3": z3
    }
    return outputs


# Initialize Model & Move to CUDA
CUDA = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = setupForData(numclasses=speciesamt, rank_classes_dict=rank_classes).to(CUDA)

# loss function and optimization
critce = neuralnetwork.CrossEntropyLoss(label_smoothing=0.1)
critce_rank = neuralnetwork.CrossEntropyLoss(label_smoothing=0.1, ignore_index=-1)
ortholossfn = OrthogonalStreamContrastiveLoss(alpha=0.1)

optimizer_groups = build_optimizer_param_groups(
    model,
    model.path1ViT,
    base_lr=1e-5,
    decay=0.75,
    weight_decay=0.01
)
optimizer = SAM(
    optimizer_groups,
    torch.optim.AdamW,
    rho=0.05,
    lr=1e-5,
    weight_decay=0.01
)
scaler = (
    torch.amp.GradScaler("cuda")
    if CUDA.type == "cuda"
    else torch.amp.GradScaler("cpu")
)

# training loop
epochsnum = 25
bestVALacc = 0.0

print(f"Training on {CUDA}")

for epoch in range(epochsnum):
  model.train()
  runningloss = 0.0
  correctpredictions = 0
  totalpredictions = 0

  for batch in loader:
    macroIN2 = batch["macro"].to(CUDA)
    microIN2 = batch["micro"].to(CUDA)
    spatialIN2 = batch["spatial"].to(CUDA)
    labels = batch["label"].to(CUDA)

    optimizer.zero_grad(set_to_none=True)

    with torch.amp.autocast("cuda" if CUDA.type == "cuda" else "cpu"):
      outputs = model(macroIN2, microIN2, spatialIN2)
      lossspecies = critce(outputs["species"], labels)

      losshierarchy = 0.0
      for rank in targetranks:
        ranktarg = [HIERARCHY.get(l, {}).get(rank, -1) for l in labels.tolist()]
        ranktensor = torch.tensor(ranktarg, dtype=torch.long, device=CUDA)
        weight = 0.5 if rank == "genus" else (0.25 if rank == "family" else 0.15)
        losshierarchy += weight * critce_rank(outputs[rank], ranktensor)

      lossortho = ortholossfn(outputs["z1"], outputs["z2"], outputs["z3"])
      loss = lossspecies + losshierarchy + lossortho

    if not torch.isfinite(loss).all():
      print("[WARNING] Non-finite loss detected in initial pass! Skipping batch.")
      optimizer.zero_grad(set_to_none=True)
      continue

    scaler.scale(loss).backward()
    optimizer.first_step(zero_grad=True)

    with torch.amp.autocast("cuda" if CUDA.type == "cuda" else "cpu"):
      outputs_perturbed = model(macroIN2, microIN2, spatialIN2)
      lossspecies_p = critce(outputs_perturbed["species"], labels)

      losshierarchy_p = 0.0
      for rank in targetranks:
        ranktarg = [HIERARCHY.get(l, {}).get(rank, -1) for l in labels.tolist()]
        ranktensor = torch.tensor(ranktarg, dtype=torch.long, device=CUDA)
        weight = 0.5 if rank == "genus" else (0.25 if rank == "family" else 0.15)
        losshierarchy_p += weight * critce_rank(outputs_perturbed[rank], ranktensor)

      lossortho_p = ortholossfn(
          outputs_perturbed["z1"],
          outputs_perturbed["z2"],
          outputs_perturbed["z3"]
      )
      loss_perturbed = lossspecies_p + losshierarchy_p + lossortho_p

    if not torch.isfinite(loss_perturbed).all():
      print("[WARNING] Non-finite loss detected in perturbed pass! Restoring weights.")
      for group in optimizer.param_groups:
        for p in group["params"]:
          old_p = optimizer.state[p].pop("old_p", None)
          if old_p is not None:
            p.copy_(old_p)
      optimizer.zero_grad(set_to_none=True)
      continue

    scaler.scale(loss_perturbed).backward()
    scaler.unscale_(optimizer.base_optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.second_step(zero_grad=True)
    scaler.update()

    runningloss += loss.item() * labels.size(0)
    _, predicts = torch.max(outputs["species"], dim=1)
    correctpredictions += torch.sum(predicts == labels).item()
    totalpredictions += labels.size(0)

  epochloss = runningloss / max(1, totalpredictions)
  epochaccuracy = (correctpredictions / max(1, totalpredictions)) * 100

  # VALIDATION TESTING
  model.eval()
  VALrunningloss = 0.0
  VALcorrect = 0
  VALtotal = 0

  with torch.no_grad():
    for valbatch in validationL:
      valMAC = valbatch["macro"].to(CUDA)
      valMIC = valbatch["micro"].to(CUDA)
      valSPA = valbatch["spatial"].to(CUDA)
      vallabels = valbatch["label"].to(CUDA)

      with torch.amp.autocast("cuda" if CUDA.type == "cuda" else "cpu"):
        VALoutputs = model(valMAC, valMIC, valSPA)
        VALloss = critce(VALoutputs["species"], vallabels)

      if not torch.isfinite(VALloss).all():
        continue

      VALrunningloss += VALloss.item() * vallabels.size(0)
      _, VALpredictions = torch.max(VALoutputs["species"], dim=1)
      VALcorrect += torch.sum(VALpredictions == vallabels).item()
      VALtotal += vallabels.size(0)

  VALloss2 = VALrunningloss / max(1, VALtotal)
  VALaccuracy = (VALcorrect / max(1, VALtotal)) * 100

  if VALaccuracy > bestVALacc:
    bestVALacc = VALaccuracy
    save_file(model.state_dict(), "convnextDINOSAUR_best.safetensors")
    print(f"New best model saved with Val Accuracy: {VALaccuracy:.2f}%")

  print(f"Epoch [{epoch+1}/{epochsnum}] - Training Loss: {epochloss:.4f} | "
        f"Train Acc: {epochaccuracy:.2f}% | Validation Loss: {VALloss2:.4f} | "
        f"Validation Accuracy: {VALaccuracy:.2f}%")


# POST-HOC TEMPERATURE SCALING
class TemperatureScaler(neuralnetwork.Module):
  def __init__(self):
    super().__init__()
    self.log_temperature = neuralnetwork.Parameter(torch.zeros(()))

  @property
  def temperature(self):
    return self.log_temperature.exp().clamp(0.05, 100.0)

  def forward(self, logits):
    return logits / self.temperature

def inverse_frequency_weights(labels, num_classes, valid_mask=None):
  labels = labels.clone().long()
  if valid_mask is not None:
    labels = labels[valid_mask]
  if len(labels) == 0:
    return None

  counts = torch.bincount(labels, minlength=num_classes).float()
  counts = torch.clamp(counts, min=1.0)
  weights = counts.sum() / (num_classes * counts)
  weights = weights / weights.mean()
  return weights

def collect_calibration_logits(model, dataloader, hierarchy, device):
  model.eval()
  collected = defaultdict(list)

  with torch.no_grad():
    for batch in dataloader:
      valMAC = batch["macro"].to(device)
      valMIC = batch["micro"].to(device)
      valSPA = batch["spatial"].to(device)
      specieslabels = batch["label"].to(device)

      with torch.amp.autocast("cuda" if device.type == "cuda" else "cpu"):
        outputs = model(valMAC, valMIC, valSPA)

      collected["species_logits"].append(outputs["species"].cpu())
      collected["species_labels"].append(specieslabels.cpu())

      species_cpu = specieslabels.cpu().tolist()
      for rank in targetranks:
        ranklabels = torch.tensor(
            [hierarchy.get(l, {}).get(rank, -1) for l in species_cpu],
            dtype=torch.long
        )
        collected[f"{rank}_logits"].append(outputs[rank].cpu())
        collected[f"{rank}_labels"].append(ranklabels)

  return {
      key: torch.cat(values, dim=0)
      for key, values in collected.items()
  }

def fit_temperature(logits, labels, weights=None, valid_mask=None):
  logits = logits.detach().float().cpu()
  labels = labels.detach().long().cpu()

  if valid_mask is None:
    valid_mask = torch.ones(labels.shape[0], dtype=torch.bool)

  logits = logits[valid_mask]
  labels = labels[valid_mask]
  if len(labels) == 0:
    return 1.0

  if weights is None:
    weights = inverse_frequency_weights(
        labels,
        logits.shape[1],
        valid_mask=torch.ones_like(valid_mask)
    )
  if weights is None:
    weights = torch.ones(logits.shape[1], dtype=torch.float32)

  weights = weights.to(logits.device)
  sample_weights = weights[labels]
  sample_weights = sample_weights / sample_weights.mean()

  scaler = TemperatureScaler()
  lbfgs = torch.optim.LBFGS(
      scaler.parameters(),
      lr=0.5,
      max_iter=100,
      tolerance_grad=1e-7,
      tolerance_change=1e-9,
      line_search_fn="strong_wolfe"
  )
  criterion = neuralnetwork.CrossEntropyLoss(reduction="none")

  def closure():
    lbfgs.zero_grad()
    losses = criterion(scaler(logits), labels)
    loss = torch.sum(losses * sample_weights) / torch.sum(sample_weights)
    loss.backward()
    return loss

  lbfgs.step(closure)
  temperature = float(scaler.temperature.detach().item())
  if not nump.isfinite(temperature):
    return 1.0
  return temperature

calibration_data = collect_calibration_logits(
    model,
    calibrationL,
    HIERARCHY,
    CUDA
)

temperatures = {}
head_names = ["species"] + targetranks
for head_name in head_names:
  logits = calibration_data[f"{head_name}_logits"]
  labels = calibration_data[f"{head_name}_labels"]
  valid_mask = labels >= 0

  weights = inverse_frequency_weights(
      labels,
      logits.shape[1],
      valid_mask=valid_mask
  )
  temperatures[head_name] = fit_temperature(
      logits,
      labels,
      weights=weights,
      valid_mask=valid_mask
  )

print("Post-hoc temperature scaling:", temperatures)

def calibrate_outputs(raw_outputs, temperatures):
  calibrated = {}
  for head_name in ["species"] + targetranks:
    temperature = float(temperatures[head_name])
    calibrated[head_name] = torch.softmax(
        raw_outputs[head_name] / temperature,
        dim=-1
    )
  return calibrated

def calibrated_species_guess(raw_outputs, temperatures, txnID):
  calibrated = calibrate_outputs(raw_outputs, temperatures)
  confidence, species_index = calibrated["species"].max(dim=-1)
  species_name = (
      fulltrainset.classes[species_index.item()]
      if species_index.item() < len(fulltrainset.classes)
      else None
  )
  return guessing(species_name, confidence.item(), txnID), calibrated


# SAVE FINAL WEIGHTS AFTER ALL EPOCHS FINISH
save_file(model.state_dict(), "convnextDINOSAUR.safetensors")

metadata = {
    "temperatures": temperatures,
    "class_names": fulltrainset.classes,
    "rank_vocabularies": {
        rank: {
            name: index
            for name, index in rankvocabularies[rank].items()
        }
        for rank in targetranks
    },
    "confidence_level": list(confidencelevel)
}
with open("convnextDINOSAUR_metadata.json", "w") as metadata_file:
  json.dump(metadata, metadata_file)

cowsay.trex("saved to convnextDINOSAUR.safetensors")

In [ ]:
# Inference script

import torch
from torchvision import transforms
from PIL import Image
import json
import os
from safetensors.torch import load_file
from google.colab import files

# Load class names
with open("class_names.json", "r") as file:
    class_names = json.load(file)

speciesamt = len(class_names)

# Initialize model architecture structure
CUDA = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inferencemodel = setupForData(numclasses=speciesamt, rank_classes_dict=rank_classes).to(CUDA)

weights_path = "convnextDINOSAUR_best.safetensors"
if os.path.exists(weights_path):
    state_dict = load_file(weights_path)
    inferencemodel.load_state_dict(state_dict)
    print(f"Successfully loaded checkpoint: {weights_path}")
else:
    print(f"{weights_path} not found.")

inferencemodel.eval()

# Define Pristine & Constrained TTA Transforms for Inference
inferencemacro = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

inferencemicro = transforms.Compose([
    transforms.Resize((256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

inferencespatial = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Production Inference & PBDB Fallback Execution Function
def predictSPEC(image_path):
    if not os.path.exists(image_path):
        print(f"Image path {image_path} does not exist.")
        return None

    raw = Image.open(image_path).convert("RGB")

    macro = inferencemacro(raw).unsqueeze(0).to(CUDA)
    micro = inferencemicro(raw).unsqueeze(0).to(CUDA)
    spatial = inferencespatial(raw).unsqueeze(0).to(CUDA)

    with torch.no_grad():
        outputs = inferencemodel(macro, micro, spatial)

        # Species probabilities via Softmax over species head (including +1 Unknown Clade logit)
        logits = outputs["species"]
        probs = torch.softmax(logits, dim=1)
        confscore, predictionindex = torch.max(probs, dim=1)

        confscore = confscore.item()
        predictionindex = predictionindex.item()

        # Check for Open-Set Recognition "Unknown Clade" trigger
        if predictionindex >= len(class_names):
            print("\n[OOD ALERT] Specimen prediction landed on Unknown Clade logit buffer.")
            print("Flagged as an uncataloged or undescribed fossil discovery!")
            return {"status": "NEW_DISCOVERY", "confidence": confscore}

        predictedspeciesraw = class_names[predictionindex]
        formattedspeciesname = predictedspeciesraw.replace("_", " ")
        matchedID = None
        for tid, rec in dinotax2.items():
            if rec["name"] and rec["name"].strip().lower() == formattedspeciesname.lower():
                matchedID = tid
                break

        print("\n--- FIV1 FOSSIL IDENTIFICATION REPORT ---")
        print(f"Raw Model Confidence: {confscore * 100:.2f}%")
        print(f"Top Species Prediction: {formattedspeciesname}")

        finalname, finalscore, taxonomicrank = guessing(formattedspeciesname, confscore, matchedID)

        return {
            "identified_name": finalname,
            "confidence": finalscore,
            "rank": taxonomicrank,
            "raw_logits": outputs
        }


print("Please upload a fossil specimen image for identification:")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\nProcessing uploaded image: {filename}")
    result = predictSPEC(filename)

In [ ]:
# The following cells are for pruning bad folders (aka genera) from the dataset, and restoring them if one is deleted by mistake.

In [ ]:
# for deleting

import os
import shutil
import kagglehub
from google.colab import userdata

remove_list = [                   # enter the names of the genuses you want to remove
    "",

]

for dino in remove_list:
    folder_name = dino.replace(" ", "_")
    folder_path = os.path.join("./fossil-data", folder_name)
    if os.path.exists(folder_path):
        shutil.rmtree(folder_path)
        print(f"Removed local folder: {folder_name}")

kUser = userdata.get('KAGGLE_USERNAME')
kSlug = userdata.get('KAGGLE_SLUG')
dataDir = "./fossil-data"

handle = f"{kUser}/{kSlug}"
print(f"Pushing updated dataset to: {handle}")
kagglehub.dataset_upload(
    handle=handle,
    local_dataset_dir=dataDir,
    version_notes="Pruned bad species"
)
print("Upload complete!")

In [ ]:
# for restoring

import os
import shutil
import kagglehub
from google.colab import userdata

# list all the genera you want to reset/rescrape here:
target_species = [""]

for dino in target_species:
    print(f"Processing: {dino}")
    bad_folder = os.path.join(dataDir, dino)

    # 1. remove local folder if it exists (just in case measure)
    if os.path.exists(bad_folder):
        shutil.rmtree(bad_folder)
        print(f"-> Removed {dino} from local data.")

    # 2. rescrape and filter
    print(f"-> Rescraping {dino}...")
    scrapeDino(dino, mIMperSP)
    filterIM(dino)

# 3.push the fully updated dataset to Kaggle once all are done
kUser = userdata.get('KAGGLE_USERNAME')
kSlug = userdata.get('KAGGLE_SLUG')
handle = f"{kUser}/{kSlug}"

print(f"Pushing corrected dataset to: {handle}")
kagglehub.dataset_upload(
    handle=handle,
    local_dataset_dir=dataDir,
    version_notes="new"
)
print("Upload complete")

In [ ]:
# The following cells are for debugging cell 4; use if you recieve a 401 Unauthorized error from Google Kaggle. Otherwise, use Colab secrets.

In [ ]:
import os
import shutil

os.makedirs('/root/.kaggle', exist_ok=True)
if os.path.exists('kaggle.json'):
    shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 600)
    print("New kaggle.json successfully installed!")
else:
    print("Please upload your kaggle.json file using the widget above.")

In [ ]:
from google.colab import files
files.upload()

In [ ]:
# Confusion Matrix; this will show you which species of dinosaur the model is most confused with.
# This cell outputs a graph along with the 10 most confused species pairs. Run this at the end of training.

import seaborn as sb
import matplotlib.pyplot as mpl
from sklearn.metrics import confusion_matrix
import numpy as np
import torch
from safetensors.torch import load_file
import os
import json

# Ensure CUDA and model instances are active
CUDA = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Quick check if model is defined; if not, re-initialize it
if 'model' not in globals():
    print("[INFO] Initializing model instance for evaluation...")
    with open("class_names.json", "r") as f:
        class_names_loaded = json.load(f)
    speciesamt = len(class_names_loaded)
    model = setupForData(numclasses=speciesamt, rank_classes_dict=rank_classes).to(CUDA)

# Load the best saved weights (using the correct filename)
weights_path = "convnextDINOSAUR_best.safetensors"
if os.path.exists(weights_path):
    model.load_state_dict(load_file(weights_path))
    print(f"Successfully loaded weights from {weights_path}")
else:
    print(f"Warning: {weights_path} not found. Ensure training has completed and saved weights.")

model.eval()

predictions = []
true_labels = []

print("Running validation inference for confusion matrix...")
with torch.no_grad():
    for batch in validationL:
        macroIN = batch["macro"].to(CUDA)
        microIN = batch["micro"].to(CUDA)
        spatialIN = batch["spatial"].to(CUDA)
        batchlabels = batch["label"].to(CUDA)

        # Forward pass through triple-stream architecture
        outputs = model(macroIN, microIN, spatialIN)
        species_logits = outputs["species"]

        _, preds = torch.max(species_logits, dim=1)

        # Clamp predictions to valid species range to prevent index out-of-bounds errors
        preds = torch.clamp(preds, max=len(fulltrainset.classes) - 1)

        predictions.extend(preds.cpu().numpy())
        true_labels.extend(batchlabels.cpu().numpy())

# Generate confusion matrix across valid species classes
all_class_indices = list(range(len(fulltrainset.classes)))
matrix = confusion_matrix(true_labels, predictions, labels=all_class_indices)

# Plot Heatmap
mpl.figure(figsize=(14, 12))
sb.heatmap(
    matrix,
    annot=False,
    cmap='Blues',
    fmt='g',
    xticklabels=fulltrainset.classes,
    yticklabels=fulltrainset.classes
)
mpl.title('Species Confusion Matrix (Best Model)')
mpl.ylabel('Actual Species')
mpl.xlabel('Predicted Species')
mpl.xticks(rotation=90, fontsize=8)
mpl.yticks(fontsize=8)
mpl.tight_layout()
mpl.show()

# Extract and print top confused species pairs
class_names = fulltrainset.classes
matrix_temp = matrix.copy()
np.fill_diagonal(matrix_temp, 0)
confusedindices = np.unravel_index(np.argsort(matrix_temp.ravel())[::-1], matrix_temp.shape)

print("\nTOP 10 MOST CONFUSED SPECIES PAIRS")
printed_count = 0
for i in range(len(confusedindices[0])):
    if printed_count >= 10:
        break
    actualidx = confusedindices[0][i]
    predidx = confusedindices[1][i]
    count = matrix_temp[actualidx, predidx]
    if count > 0:
        printed_count += 1
        print(f"{printed_count}. Actual: '{class_names[actualidx]}' → Mistaken for: '{class_names[predidx]}' ({count} times)")